# SFT Qwen3-0.6B on calculator tool trajectories

Fine-tune Qwen3-0.6B with LoRA on a deterministic, tier-stratified 200-row slice of the training split. The remaining 600 training IDs are reserved for GRPO. Evaluation and test data are never loaded.

Training and epoch checkpoints use fast local `/content` storage. Trackio records training health locally and persists it to an explicitly created private Hugging Face Bucket. A private live Gradio dashboard requires Hugging Face PRO, so dashboard creation failure is non-fatal. After the successful full run, the final LoRA adapter is uploaded once to a private model repository and its immutable Hub revision is recorded locally for GRPO.

Run the overfit gate first (`RUN_MODE=overfit`, the default), inspect a generated trajectory, then inject `RUN_MODE=full` into a fresh Colab CLI session and execute the notebook again for the three-epoch SFT run. Secrets and run controls come from the project `.env`, injected securely into the persistent kernel before notebook execution.

In [ ]:
# @title Upgrade the training stack as one compatible set
import importlib.metadata
import subprocess
import sys

TRAINING_PACKAGES = [
    'trl', 'transformers', 'peft', 'datasets', 'accelerate', 'trackio',
    # Colab preinstalls an old torchao; upgrade it with PEFT to avoid dispatch errors.
    'torchao',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + TRAINING_PACKAGES,
    check=True,
)
resolved = {name: importlib.metadata.version(name) for name in TRAINING_PACKAGES}
print('Resolved training stack:', resolved)

In [ ]:
# @title Authenticate from the injected .env and configure run identities
import os
from pathlib import Path

from huggingface_hub import HfApi, create_bucket, login

# With colab-cli, upload .env and run scripts/load-env.py before this notebook.
# The loader injects values into this persistent kernel and deletes /content/.env.
hf_token = os.environ.get('HF_TOKEN')
assert hf_token, 'HF_TOKEN is missing; inject the project .env before execution.'
login(token=hf_token, add_to_git_credential=False)
hf_api = HfApi(token=hf_token)
hf_user = hf_api.whoami()['name']

DATA_DIR = Path('/content/data')
DATASET_PATH = DATA_DIR / 'calculator_qwen3'
SFT_IDS_PATH = DATA_DIR / 'calculator_qwen3_sft200_ids.txt'
GRPO_IDS_PATH = DATA_DIR / 'calculator_qwen3_grpo600_ids.txt'

MODEL_ID = 'Qwen/Qwen3-0.6B'
MODEL_REVISION = hf_api.model_info(MODEL_ID).sha  # freeze exact base weights
SEED = 42  # frozen for split construction, Trainer seed, and data seed
SFT_SIZE = 200
RUN_MODE = os.environ.get('RUN_MODE', 'overfit')  # inject 'full' only after gate passes
OVERFIT_ROWS = int(os.environ.get('OVERFIT_ROWS', '16'))
USE_PRECOMPUTED_IDS = os.environ.get('USE_PRECOMPUTED_IDS', '0') == '1'
SFT_MAX_LENGTH = int(os.environ.get('SFT_MAX_LENGTH', '1024'))
assert SFT_MAX_LENGTH >= 1024
RUN_TAG = os.environ.get('RUN_TAG', '').strip()
assert not RUN_TAG or RUN_TAG.replace('-', '').replace('_', '').isalnum(), RUN_TAG
RUN_SUFFIX = f'-{RUN_TAG}' if RUN_TAG else ''
EXPERIMENT_ROOT = Path(f'/content/experiments/qwen3-calc-sft200-seed42{RUN_SUFFIX}')

HUB_REPO_ID = os.environ.get('HUB_REPO_ID', f'{hf_user}/qwen3-0.6b-calc-sft200')
TRACKIO_PROJECT = os.environ.get('TRACKIO_PROJECT', 'calc-rlvr-sft')
TRACKIO_SPACE_ID = os.environ.get('TRACKIO_SPACE_ID', f'{hf_user}/calc-rlvr-sft')
TRACKIO_BUCKET_ID = os.environ.get(
    'TRACKIO_BUCKET_ID', f'{hf_user}/calc-rlvr-sft-bucket'
)
RUN_NAME = f'qwen3-calc-sft200-{RUN_MODE}-seed{SEED}{RUN_SUFFIX}'

DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
assert DATASET_PATH.is_dir(), f'Upload the HF dataset directory to {DATASET_PATH}'
assert RUN_MODE in {'overfit', 'full'}
# Auto-created Trackio Buckets previously returned 404. Create and validate the
# private bucket explicitly so failure occurs before expensive GPU training.
bucket = create_bucket(
    TRACKIO_BUCKET_ID, private=True, exist_ok=True, token=hf_token,
)
assert bucket.bucket_id == TRACKIO_BUCKET_ID
print({
    'run_name': RUN_NAME, 'model_revision': MODEL_REVISION,
    'hub_repo': HUB_REPO_ID, 'trackio_space': TRACKIO_SPACE_ID,
    'trackio_bucket': TRACKIO_BUCKET_ID,
})

In [ ]:
# @title Create and persist the tier-stratified SFT/GRPO split
import random
from collections import defaultdict

from datasets import load_from_disk

def remove_null_fields(value):
    if isinstance(value, dict):
        return {key: remove_null_fields(child) for key, child in value.items() if child is not None}
    if isinstance(value, list):
        return [remove_null_fields(child) for child in value]
    return value

hf_dataset = load_from_disk(str(DATASET_PATH))
train_rows = [remove_null_fields(dict(row)) for row in hf_dataset['train']]

assert len(train_rows) == 800, f'Expected 800 training rows, found {len(train_rows)}'
assert len({row['id'] for row in train_rows}) == len(train_rows), 'Training IDs must be unique'

by_tier = defaultdict(list)
for row in train_rows:
    by_tier[row['metadata']['tier']].append(row)

if USE_PRECOMPUTED_IDS:
    sft_ids = sorted(
        line.strip() for line in SFT_IDS_PATH.read_text().splitlines() if line.strip()
    )
    grpo_ids = sorted(
        line.strip() for line in GRPO_IDS_PATH.read_text().splitlines() if line.strip()
    )
    sft_id_set = set(sft_ids)
    sft_rows = sorted(
        (row for row in train_rows if row['id'] in sft_id_set),
        key=lambda row: row['id'],
    )
else:
    # Largest-remainder allocation preserves tier proportions and totals exactly 200.
    raw_quotas = {tier: len(rows) * SFT_SIZE / len(train_rows) for tier, rows in by_tier.items()}
    quotas = {tier: int(quota) for tier, quota in raw_quotas.items()}
    for tier in sorted(raw_quotas, key=lambda key: (raw_quotas[key] - quotas[key], key), reverse=True)[:SFT_SIZE - sum(quotas.values())]:
        quotas[tier] += 1

    rng = random.Random(SEED)
    sft_rows = []
    for tier in sorted(by_tier):
        tier_rows = sorted(by_tier[tier], key=lambda row: row['id'])
        rng.shuffle(tier_rows)
        sft_rows.extend(tier_rows[:quotas[tier]])

    sft_rows.sort(key=lambda row: row['id'])
    sft_ids = [row['id'] for row in sft_rows]
    grpo_ids = sorted({row['id'] for row in train_rows} - set(sft_ids))
    SFT_IDS_PATH.write_text('\n'.join(sft_ids) + '\n')
    GRPO_IDS_PATH.write_text('\n'.join(grpo_ids) + '\n')

assert len(sft_ids) == SFT_SIZE
assert len(grpo_ids) == len(train_rows) - SFT_SIZE
assert not set(sft_ids) & set(grpo_ids)
assert set(sft_ids) | set(grpo_ids) == {row['id'] for row in train_rows}

print('SFT rows by tier:', {tier: sum(row['metadata']['tier'] == tier for row in sft_rows) for tier in sorted(by_tier)})
print(f'Wrote {SFT_IDS_PATH} and {GRPO_IDS_PATH}')

In [ ]:
# @title Load only SFT fields and verify Qwen's tool chat template
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

sft_dataset = Dataset.from_list([
    {'messages': row['messages'], 'tools': row['tools']}
    for row in sft_rows
])
rendered = tokenizer.apply_chat_template(
    sft_dataset[0]['messages'], tools=sft_dataset[0]['tools'], tokenize=False
)
assert '<think>' in rendered and 'calculator' in rendered
print(rendered[:1_500])

In [ ]:
# @title Measure sequence lengths (SFT-200 only)
def token_length(example):
    rendered = tokenizer.apply_chat_template(
        example['messages'], tools=example['tools'], tokenize=False,
        add_generation_prompt=False,
    )
    return len(tokenizer(rendered, add_special_tokens=False)['input_ids'])

lengths = [token_length(example) for example in sft_dataset]
length_summary = {
    'min': min(lengths),
    'p50': sorted(lengths)[len(lengths) // 2],
    'max': max(lengths),
}
print(length_summary)
assert 400 <= min(lengths) <= max(lengths) <= SFT_MAX_LENGTH, (
    'Unexpected or truncated Qwen chat-template token lengths; inspect before training.',
    length_summary,
    {'sft_max_length': SFT_MAX_LENGTH},
)

In [ ]:
# @title Build LoRA SFT trainer with Trackio (no evaluation dataset)
import math

import torch
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required. Select a T4 or L4 GPU in Colab.')

gpu_name = torch.cuda.get_device_name(0)
compute_capability = torch.cuda.get_device_capability(0)
# T4 (7.5) has no native BF16 tensor cores; use FP16 below Ampere (8.0).
use_bf16 = compute_capability[0] >= 8 and torch.cuda.is_bf16_supported()
dtype_name = 'bfloat16' if use_bf16 else 'float16'
print(f'GPU: {gpu_name}; capability: {compute_capability}; dtype: {dtype_name}')

train_dataset = sft_dataset.select(range(OVERFIT_ROWS)) if RUN_MODE == 'overfit' else sft_dataset
epochs = 20 if RUN_MODE == 'overfit' else 3
optimizer_steps = math.ceil(len(train_dataset) / (2 * 4)) * epochs
warmup_steps = max(1, round(optimizer_steps * 0.05))
run_output_dir = EXPERIMENT_ROOT / RUN_MODE
run_output_dir.mkdir(parents=True, exist_ok=True)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
sft_config = SFTConfig(
    output_dir=str(run_output_dir), seed=SEED, data_seed=SEED,
    max_length=SFT_MAX_LENGTH, packing=False, assistant_only_loss=True, loss_type='chunked_nll',
    shuffle_dataset=True, num_train_epochs=epochs,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=1e-4, warmup_steps=warmup_steps, lr_scheduler_type='cosine',
    weight_decay=0.0, max_grad_norm=1.0,
    bf16=use_bf16, fp16=not use_bf16, gradient_checkpointing=True,
    do_eval=False, eval_strategy='no', logging_steps=5,
    save_strategy='epoch', save_total_limit=2,
    report_to='trackio', project=TRACKIO_PROJECT, run_name=RUN_NAME,
    trackio_space_id=TRACKIO_SPACE_ID, trackio_bucket_id=TRACKIO_BUCKET_ID,
    trackio_static_space_id=False, hub_private_repo=True,
    model_init_kwargs={'dtype': dtype_name, 'revision': MODEL_REVISION},
)
trainer = SFTTrainer(
    model=MODEL_ID, args=sft_config, train_dataset=train_dataset,
    processing_class=tokenizer, peft_config=peft_config,
)
print({
    'mode': RUN_MODE, 'rows': len(train_dataset), 'epochs': epochs,
    'output': str(run_output_dir), 'trackio': TRACKIO_SPACE_ID,
})

In [ ]:
# @title Train, validate logs, save locally, and publish the full adapter once
import hashlib
import importlib.metadata
import json
import math
import platform
import shutil
from datetime import datetime, timezone


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def json_default(value):
    if isinstance(value, set):
        return sorted(value)
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, 'value'):
        return value.value
    return str(value)


def write_json_atomic(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(
        payload, indent=2, sort_keys=True, default=json_default,
    ) + '\n')
    tmp.replace(path)


started_at = datetime.now(timezone.utc).isoformat()
train_result = trainer.train()
ended_at = datetime.now(timezone.utc).isoformat()
trainer.save_state()

# Verify that Trackio/Trainer received finite training-health metrics.
log_history = trainer.state.log_history
for entry in log_history:
    for key, value in entry.items():
        if isinstance(value, (int, float)) and not math.isfinite(value):
            raise RuntimeError(f'Non-finite metric: {key}={value}')

observed_keys = {key for entry in log_history for key in entry}
expected_log_keys = {'loss', 'learning_rate', 'grad_norm', 'epoch', 'mean_token_accuracy', 'entropy'}
missing_keys = expected_log_keys - observed_keys
if missing_keys:
    print('WARNING: expected metrics not emitted by this TRL run:', sorted(missing_keys))
losses = [entry['loss'] for entry in log_history if 'loss' in entry]
assert losses, 'Trainer did not emit training loss.'
print({
    'first_logged_loss': losses[0], 'last_logged_loss': losses[-1],
    'log_points': len(losses), 'observed_metric_keys': sorted(observed_keys),
    'trackio_url': f'https://huggingface.co/spaces/{TRACKIO_SPACE_ID}',
})

# Keep checkpoints in run_output_dir, but publish only the compact final adapter.
adapter_dir = run_output_dir / 'final_adapter'
adapter_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
shutil.copy2(SFT_IDS_PATH, adapter_dir / SFT_IDS_PATH.name)
shutil.copy2(GRPO_IDS_PATH, adapter_dir / GRPO_IDS_PATH.name)

package_names = [
    'trl', 'transformers', 'peft', 'datasets', 'accelerate', 'trackio',
    'torch', 'torchao',
]
run_config = {
    'run_name': RUN_NAME,
    'started_at_utc': started_at,
    'ended_at_utc': ended_at,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'seed': SEED,
    'data_seed': SEED,
    'run_mode': RUN_MODE,
    'num_rows': len(train_dataset),
    'num_train_epochs': epochs,
    'input_sha256': {
        'dataset_dict': sha256_file(DATASET_PATH / 'dataset_dict.json'),
        'train_arrow': sha256_file(
            DATASET_PATH / 'train' / 'data-00000-of-00001.arrow'
        ),
        'sft200_ids': sha256_file(SFT_IDS_PATH),
        'grpo600_ids': sha256_file(GRPO_IDS_PATH),
    },
    'gpu_name': gpu_name,
    'dtype': dtype_name,
    'python': platform.python_version(),
    'packages': {name: importlib.metadata.version(name) for name in package_names},
    'trackio': {
        'project': TRACKIO_PROJECT,
        'space_id': TRACKIO_SPACE_ID,
        'space_url': f'https://huggingface.co/spaces/{TRACKIO_SPACE_ID}',
        'bucket_id': TRACKIO_BUCKET_ID,
        'bucket_url': f'https://huggingface.co/buckets/{TRACKIO_BUCKET_ID}',
        'dashboard_status': 'requires HF PRO for private Gradio Space',
    },
    'sft_config': sft_config.to_dict(),
    'peft_config': peft_config.to_dict(),
    'train_metrics': train_result.metrics,
    'hub_repo_id': HUB_REPO_ID if RUN_MODE == 'full' else None,
    'hub_revision': None,
}
config_path = adapter_dir / 'sft_run_config.json'
write_json_atomic(config_path, run_config)

if RUN_MODE == 'full':
    # Trackio's live process can alter Hub upload state. Generate a helper that
    # must be run only after restarting the kernel and reinjecting HF_TOKEN.
    upload_script = Path('/content/upload_sft_adapter.py')
    upload_script.write_text(f'''import json
import os
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
from huggingface_hub import HfApi

repo_id = {HUB_REPO_ID!r}
adapter_dir = Path({str(adapter_dir)!r})
run_output_dir = Path({str(run_output_dir)!r})
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
commit = api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=str(adapter_dir),
    commit_message="Qwen3 calculator SFT-200 seed {SEED}",
)
revision = commit.oid
manifest = {{
    "repo_id": repo_id,
    "revision": revision,
    "url": f"https://huggingface.co/{{repo_id}}/tree/{{revision}}",
}}
(run_output_dir / "hub_revision.json").write_text(
    json.dumps(manifest, indent=2) + "\\n"
)
print("UPLOAD_COMPLETE", manifest)
''')
    print('After this notebook: restart kernel, reinject .env, then run:')
    print(f'colab exec -s <session> -f "{upload_script}" --timeout 900')
else:
    print('Overfit adapter remains local and is not published as the GRPO base.')

print(train_result.metrics)
print(f'Saved final adapter and run manifest to {adapter_dir}')

In [ ]:
# @title Overfit gate: run one calculator tool loop on an SFT row
import re

def calculator(op, a, b):
    if op == '+': return a + b
    if op == '-': return a - b
    if op == '*': return a * b
    raise ValueError(f'Unsupported operator: {op}')

record = sft_rows[0]
messages = [dict(message) for message in record['messages'][:2]]
tool_pattern = re.compile(r'<tool_call>\s*(.*?)\s*</tool_call>', re.DOTALL)
model = trainer.model.eval()

for turn in range(6):
    prompt = tokenizer.apply_chat_template(
        messages, tools=record['tools'], tokenize=False,
        add_generation_prompt=True, enable_thinking=True,
    )
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            stop_strings=['</tool_call>'], tokenizer=tokenizer,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print(f'--- assistant turn {turn + 1} ---\n{text}')

    calls = tool_pattern.findall(text)
    if not calls:
        break  # Final assistant answer.
    assert len(calls) == 1, f'Expected one tool call, found {len(calls)}'
    payload = json.loads(calls[0])
    assert payload['name'] == 'calculator'
    args = payload['arguments']
    result = calculator(**args)
    call_id = f'gate_call_{turn + 1:02d}'
    messages.extend([
        {'role': 'assistant', 'content': text.split('<tool_call>', 1)[0].strip(), 'tool_calls': [
            {'id': call_id, 'type': 'function', 'function': {'name': 'calculator', 'arguments': args}}
        ]},
        {'role': 'tool', 'name': 'calculator', 'tool_call_id': call_id, 'content': json.dumps({'result': result})},
    ])
    print('tool result:', result)
else:
    raise RuntimeError('Tool loop exceeded six turns.')

## Gate before the full run

After the overfit run:

1. Inspect the preceding cell for thinking, exactly one calculator call per assistant tool turn, tool result, and continuation. Do not use eval or test rows.
2. Confirm finite loss/gradient metrics and a decreasing loss trend in Trainer output/state; the overfit run should strongly fit its 16 rows. Trackio also persists logs to the printed private Bucket. A private live Space is unavailable without Hugging Face PRO.
3. Start a fresh CLI session with `RUN_MODE=full` injected through `.env`, then execute the whole notebook. The full run uses a separate Trackio run name.
4. After the full notebook finishes, restart that kernel, reinject `.env`, and execute `/content/upload_sft_adapter.py`. This isolates the final Hub upload from Trackio's live process.
5. Record the printed `repo_id` and immutable `revision` in `TODO.md`. GRPO must load that exact revision.

`/content` is ephemeral. Use `colab download` on the lightweight bundle and `hub_revision.json` before releasing the runtime.

In [ ]:
# @title Bundle lightweight artifacts for colab-cli download
import zipfile

bundle_path = Path(f'/content/{RUN_NAME}-artifacts.zip')
artifact_paths = [
    DATASET_PATH / 'dataset_dict.json',
    DATASET_PATH / 'train' / 'data-00000-of-00001.arrow',
    SFT_IDS_PATH,
    GRPO_IDS_PATH,
    adapter_dir / 'sft_run_config.json',
    run_output_dir / 'trainer_state.json',
]
revision_path = run_output_dir / 'hub_revision.json'
if revision_path.exists():
    artifact_paths.append(revision_path)
upload_script = Path('/content/upload_sft_adapter.py')
if upload_script.exists():
    artifact_paths.append(upload_script)

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_paths:
        assert path.is_file(), f'Missing required artifact: {path}'
        archive.write(path, arcname=path.name)

print(f'Created {bundle_path} ({bundle_path.stat().st_size:,} bytes)')
print('Download before stopping the VM:')
print(f'colab download -s <session> "{bundle_path}" "outputs/{bundle_path.name}"')

## Post-SFT evaluation (fresh inference process)

Evaluate the exact published adapter only after its immutable Hub revision is recorded. Use `scripts/evaluate_sft_qwen3_calculator.py`; it loads the pinned base and LoRA revisions, calls `merge_and_unload(safe_merge=True)`, verifies that no PEFT wrapper remains, then runs the strict thinking-mode tool loop on the HF `eval` split for three fixed seeds. The test split remains untouched.

Run this in a fresh T4 session so Trainer/Trackio state does not consume memory or alter Hub behavior:

```bash
colab new -s <session> --gpu T4
echo "from pathlib import Path; Path('/content/data').mkdir(parents=True, exist_ok=True)" | colab exec -s <session>
colab upload -s <session> scripts/evaluate_sft_qwen3_calculator.py /content/evaluate_sft_qwen3_calculator.py
colab upload -s <session> data/calculator_qwen3 /content/data/calculator_qwen3
colab upload -s <session> .env /content/.env
colab exec -s <session> -f .cursor/skills/colab-cli/scripts/load-env.py --timeout 60
echo "import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','transformers','peft','accelerate','tqdm','torchao'],check=True)" | colab exec -s <session> --timeout 900
colab restart-kernel -s <session>
colab upload -s <session> .env /content/.env
colab exec -s <session> -f .cursor/skills/colab-cli/scripts/load-env.py --timeout 60
echo "import runpy,sys; sys.argv=['evaluate_sft_qwen3_calculator.py']; _=runpy.run_path('/content/evaluate_sft_qwen3_calculator.py',run_name='__main__')" | colab exec -s <session> --timeout 3600
```

Download `/content/experiments/qwen3-calc-sft200-eval` before releasing the runtime.